<a href="https://colab.research.google.com/github/AnanyaAsthana/Hadoop-CUDA-Lab/blob/main/Ques2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile wordcount.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda.h>
#include <time.h>

#define MAX_WORDS 200
#define MAX_LEN 20

void cpu_word_count(char words[][MAX_LEN], int n) {
    int count[MAX_WORDS] = {0};
    int visited[MAX_WORDS] = {0};

    printf("\nCPU Word Count:\n");

    for (int i = 0; i < n; i++) {
        if (visited[i]) continue;

        int c = 1;
        for (int j = i + 1; j < n; j++) {
            if (strcmp(words[i], words[j]) == 0) {
                c++;
                visited[j] = 1;
            }
        }
        printf("%s -> %d\n", words[i], c);
    }
}

__global__ void word_count_kernel(char *words, int *counts, int n) {
    int i = threadIdx.x;

    if (i >= n) return;

    counts[i] = 1;

    for (int j = 0; j < n; j++) {
        if (i != j) {
            int match = 1;

            for (int k = 0; k < MAX_LEN; k++) {
                if (words[i*MAX_LEN + k] != words[j*MAX_LEN + k]) {
                    match = 0;
                    break;
                }
            }

            if (match) {
                atomicAdd(&counts[i], 1);
            }
        }
    }
}


int main() {


    char input[] =
    "apple banana orange apple mango banana apple grape mango peach banana apple orange mango grape peach apple banana orange mango apple banana grape peach mango apple orange banana mango grape apple banana peach orange mango apple grape banana mango apple peach orange banana apple mango grape orange banana apple mango peach grape orange banana apple mango grape peach orange banana apple mango grape peach orange banana apple mango grape peach orange banana apple mango grape peach orange banana apple mango grape peach orange banana apple mango grape peach orange banana apple mango grape peach blue yellow pink red green purple black white grey violet brown blue yellow pink red green purple black white grey violet brown blue yellow pink red green purple black white grey violet brown blue yellow pink red green purple black white grey violet brown blue yellow pink red green purple black white grey violet brown blue yellow pink red green purple black white grey violet brown blue yellow pink red green purple black white grey violet brown";

    char words[MAX_WORDS][MAX_LEN];
    int n = 0;

    // Tokenize
    char *token = strtok(input, " ");
    while (token != NULL && n < MAX_WORDS) {
        strcpy(words[n++], token);
        token = strtok(NULL, " ");
    }


    clock_t start_cpu = clock();
    cpu_word_count(words, n);
    clock_t end_cpu = clock();

    double cpu_time = (double)(end_cpu - start_cpu) / CLOCKS_PER_SEC;

    char *d_words;
    int *d_counts;

    cudaMalloc(&d_words, n * MAX_LEN * sizeof(char));
    cudaMalloc(&d_counts, n * sizeof(int));

    cudaMemcpy(d_words, words, n * MAX_LEN * sizeof(char), cudaMemcpyHostToDevice);


    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    word_count_kernel<<<1, n>>>(d_words, d_counts, n);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time, start, stop);

    int counts[MAX_WORDS];
    cudaMemcpy(counts, d_counts, n * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nGPU Word Count:\n");

    for (int i = 0; i < n; i++) {
        int printed = 0;

        for (int j = 0; j < i; j++) {
            if (strcmp(words[i], words[j]) == 0) {
                printed = 1;
                break;
            }
        }

        if (!printed) {
            printf("%s -> %d\n", words[i], counts[i]);
        }
    }

    printf("\nCPU Time: %f seconds\n", cpu_time);
    printf("GPU Time: %f ms\n", gpu_time);

    cudaFree(d_words);
    cudaFree(d_counts);

    return 0;
}

Overwriting wordcount.cu


In [ ]:
!nvcc wordcount.cu -o wordcount
!./wordcount

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
wordcount.cu(12): warning #177-D: variable "count" was declared but never referenced
      int count[200] = {0};
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"


CPU Word Count:
apple -> 19
banana -> 17
orange -> 14
mango -> 17
grape -> 14
peach -> 13
blue -> 7
yellow -> 7
pink -> 7
red -> 7
green -> 7
purple -> 7
black -> 7
white -> 7
grey -> 7
violet -> 7
brown -> 7

GPU Word Count:
apple -> 5
banana -> 5
orange -> 2
mango -> 4
grape -> 2
peach -> 3
blue -> 1
yellow -> 4
pink -> 3
red -> 1
green -> 1
purple -> 1
black -> 1
white -> 1
grey -> 1
violet -> 1
brown -> 1

CPU Time: 0.000073 seconds
GPU Time: 0.289728 ms
